# TP 3 — Stabiliser puis transformer chaque token

**Objectif :** implémenter une LayerNorm, comprendre son axe de calcul, puis construire le MLP appliqué séparément à chaque token.

Vecteur E_i (dimension C) → LayerNorm (centrer, réduire) → Feed-forward (C → 4C → C) → E_i + ΔE_i (résiduel)

Contrairement à l'attention, ces opérations ne mélangent pas les positions.

**Livrable :** une LayerNorm pédagogique validée contre `nn.LayerNorm`, un MLP C → 4C → C et un test de la connexion résiduelle.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 — Coder LayerNorm

$$\hat x_{i,j} = \frac{x_{i,j} - \mu_i}{\sqrt{\sigma_i^2 + \varepsilon}}, \qquad y_{i,j} = \gamma_j \hat x_{i,j} + \beta_j$$

*i* désigne le token ; *j*, une composante de son embedding.

In [2]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(dim=-1, keepdim=True)                   # moyenne de chaque token
        xvar = x.var(dim=-1, keepdim=True, unbiased=False)     # variance de chaque token
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma * xhat + self.beta

    def parameters(self):
        return [self.gamma, self.beta]

### Questions d'interprétation

**Sur quel axe normaliser ?** Sur le dernier axe (`dim=-1`), celui des features : chaque token est normalisé à partir de ses propres composantes.

**À quoi servent ε, γ et β ?**
- ε évite une division par zéro (ou par un nombre minuscule) si toutes les composantes d'un vecteur sont presque égales.
- γ (échelle) et β (décalage) sont appris : ils permettent au réseau de retrouver l'amplitude et le centrage qui lui conviennent pour chaque feature, la normalisation ne lui impose donc pas une contrainte définitive. Initialisés à 1 et 0, ils donnent au départ une normalisation pure.

## Étape 2 — Choisir l'axe de normalisation

Prédisez quelle ligne donnera environ (0, 1) avant d'exécuter.

In [3]:
torch.manual_seed(1337)
module = LayerNorm1d(100)

x_in = torch.randn(32, 100)
x = module(x_in)

print("x[:, 0] :", x[:, 0].mean().item(), x[:, 0].std().item())   # une feature à travers 32 tokens
print("x[0, :] :", x[0, :].mean().item(), x[0, :].std().item())   # un token à travers ses 100 features

x[:, 0] : 0.14759676158428192 0.8847486972808838
x[0, :] : -2.2649764730431343e-08 1.005033254623413


In [4]:
# Validation contre l'implémentation PyTorch (on compare sur la même ENTRÉE x_in)
ref = nn.LayerNorm(100)
print(torch.allclose(module(x_in), ref(x_in), atol=1e-6))

True


| Sélection | Ce qui varie | Résultat |
|---|---|---|
| `x[:, 0]` | les 32 tokens, feature 0 | pas imposé |
| `x[0, :]` | les 100 features du token 0 | μ ≈ 0, σ ≈ 1 |

*Remarque :* `std()` utilise la version non biaisée (division par N−1), d'où un écart-type très légèrement supérieur à 1 pour `x[0, :]`.

**Attention :** dans le code de la diapositive, `x` est écrasé par sa version normalisée avant le test `allclose`. Il faut comparer les deux modules sur la même entrée (`x_in`) ; ici, comme LayerNorm est idempotente à ε près, le test passerait quand même, mais il ne vérifierait rien d'intéressant.

## Étape 3 — Construire le réseau feed-forward

Le MLP augmente temporairement la dimension, applique une non-linéarité, puis revient à la dimension de l'embedding.

In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, mult=4, act=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, mult * n_embd),   # W_up : C -> 4C
            act(),
            nn.Linear(mult * n_embd, n_embd),   # W_down : 4C -> C
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(1337)
ffwd = FeedForward(100)
delta = ffwd(torch.randn(32, 100))
print(delta.shape)

torch.Size([32, 100])


In [6]:
# Proportion d'activations positives après ReLU
torch.manual_seed(1337)
x = torch.randn(32, 100)
h = ffwd.net[1](ffwd.net[0](x))     # après W_up et ReLU, forme (32, 400)
print(h.shape)
print("Proportion d'activations > 0 :", (h > 0).float().mean().item())

torch.Size([32, 400])
Proportion d'activations > 0 : 0.49742186069488525


In [7]:
# Le MLP ne mélange pas les tokens : traiter un token seul donne le même résultat
x = torch.randn(4, 8, 100)
full = ffwd(x)
seul = ffwd(x[:, 3:4, :])
print(torch.allclose(full[:, 3:4, :], seul, atol=1e-6))

True


### Questions d'interprétation

**Quelle proportion d'activations est positive ?** Environ 50 % à l'initialisation : les pré-activations sont à peu près symétriques autour de 0, et ReLU annule toutes celles qui sont négatives.

**Pourquoi revenir à C ?** Pour que la sortie ait la même forme que l'entrée : on peut alors l'additionner au flux résiduel et empiler les blocs. L'expansion à 4C donne simplement plus de « neurones » pour calculer.

## Étapes 4 et 5 — Ajouter le résiduel

Organisation **pré-norm** : LayerNorm avant la sous-couche, puis addition résiduelle.

In [8]:
torch.manual_seed(1337)
B, T, n_embd = 4, 8, 32

layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)

x = torch.randn(B, T, n_embd)
delta = ffwd(layer_norm(x))
x_out = x + delta

print(x.shape)
print(delta.shape)
print(x_out.shape)

torch.Size([4, 8, 32])
torch.Size([4, 8, 32])
torch.Size([4, 8, 32])


### Variantes
1. Remplacer ReLU par GELU.
2. Tester C → 2C → C, puis C → 8C → C.
3. Compter les paramètres dans chaque cas.
4. Retirer l'addition résiduelle et comparer les formes.

In [9]:
# 1. GELU vs ReLU
torch.manual_seed(1337)
x = torch.randn(32, 100)
for act in (nn.ReLU, nn.GELU):
    torch.manual_seed(0)
    f = FeedForward(100, act=act)
    h = f.net[1](f.net[0](x))
    print(f"{act.__name__:5s} : sortie {tuple(f(x).shape)}, activations > 0 : {(h > 0).float().mean():.2f}, "
          f"min activation : {h.min():.3f}")

ReLU  : sortie (32, 100), activations > 0 : 0.50, min activation : 0.000
GELU  : sortie (32, 100), activations > 0 : 0.50, min activation : -0.170


In [10]:
# 2-3. Facteur d'expansion et nombre de paramètres
C = 100
for mult in (2, 4, 8):
    f = FeedForward(C, mult=mult)
    n = sum(p.numel() for p in f.parameters())
    formule = 2 * mult * C * C + mult * C + C
    print(f"C -> {mult}C -> C : {n:>7d} paramètres (formule 2·m·C² + m·C + C = {formule})")

C -> 2C -> C :   40300 paramètres (formule 2·m·C² + m·C + C = 40300)
C -> 4C -> C :   80500 paramètres (formule 2·m·C² + m·C + C = 80500)
C -> 8C -> C :  160900 paramètres (formule 2·m·C² + m·C + C = 160900)


In [11]:
# 4. Sans résiduel : la forme est la même, mais le signal d'entrée est perdu
torch.manual_seed(1337)
x = torch.randn(B, T, n_embd)
sans_residuel = ffwd(layer_norm(x))
avec_residuel = x + ffwd(layer_norm(x))
print(sans_residuel.shape, avec_residuel.shape)
print("Corrélation entrée / sortie sans résiduel :",
      torch.corrcoef(torch.stack([x.flatten(), sans_residuel.flatten()]))[0, 1].item())
print("Corrélation entrée / sortie avec résiduel :",
      torch.corrcoef(torch.stack([x.flatten(), avec_residuel.flatten()]))[0, 1].item())

torch.Size([4, 8, 32]) torch.Size([4, 8, 32])
Corrélation entrée / sortie sans résiduel : -0.01855340786278248
Corrélation entrée / sortie avec résiduel : 0.975311815738678


**GELU :** même forme de sortie ; GELU est une version lisse de ReLU qui laisse passer de petites valeurs négatives (minimum ≈ −0,17) au lieu de les annuler.

**Paramètres (C = 100) :** 2C → 40 300 ; 4C → 80 500 ; 8C → 160 900. Le nombre de paramètres est proportionnel au facteur d'expansion.

**Sans résiduel :** les formes sont identiques — l'addition n'est donc pas une question de forme — mais la sortie n'est presque plus corrélée à l'entrée : l'information d'origine doit être entièrement recalculée par la couche.

## Bilan du TP 3

- **Quelle dimension LayerNorm conserve-t-elle ?** Toutes : la forme est inchangée `(B, T, C)`. Elle recentre et réduit chaque vecteur sur son axe C.
- **Le MLP mélange-t-il les tokens ?** Non : la même transformation est appliquée indépendamment à chaque position (vérifié plus haut). Seule l'attention fait communiquer les positions.
- **À quoi sert la connexion résiduelle ?** Chaque sous-couche n'ajoute qu'une correction ΔE au vecteur existant. L'information d'entrée est préservée et le gradient traverse directement l'addition, ce qui rend l'entraînement de réseaux profonds bien plus stable.